# Ders4 - Çoktan Seçmeli Benchmark Sorularının Oluşturulması

`BenchmarkSoruHavuzuAnalizi.ipynb` çıktısı olan `benchmark_soru_havuzu.json`'daki her
soru-cevap çiftini, modelleri test edeceğimiz **çoktan seçmeli** bir soruya dönüştürüyoruz.

Her benchmark sorusu şu unsurları içerecek:
1. **Ürün bilgisi** — modele soru sorulmadan önce verilecek bağlam (kısa, buyer-facing spec).
2. **Soru** — alıcının sorusu. Gerekiyorsa, cevaptaki bilgi ürün bilgisinde YOKSA (ör. kargo
   firması, koli/desi, spesifik bir parça ölçüsü gibi satıcıya özgü ama spekte yer almayan
   bilgiler), bu bilgi alıcının zaten bir yerden öğrenmiş/duymuş gibi doğal bir cümleyle
   sorunun içine gömülür (`soru_final`). Böylece test edilen modelin, elinde olmayan bir
   bilgiyi tahmin etmesi/uydurması gerekmez; sadece verilen bağlamı (ürün bilgisi + soru)
   doğru işleyip işlemediği ölçülür.
3. **Şıklar (A-D)** — biri doğru, üçü yanlış; doğru şık rastgele bir harfe yerleştirilir.
4. **Üslup kriteri** — şıklar sadece doğru/yanlış bilgiyle değil, **kibarlık ve müşteriyi
   olumsuz bir cevapta bile ikna etme/elde tutma çabasıyla** da ayrışır. Yani "doğru ama kaba"
   bir cevap da bir tuzak şık olarak sunulur; gerçek doğru şık hem doğru hem de nazik/ikna
   edici olmalıdır.


In [ ]:
## Gerekli paketleri kuruyoruz (ilk çalıştırmada bir kere yeterli).
!uv pip install -q pandas httpx


In [ ]:
import asyncio
import json
import os
import random
import re

import pandas as pd


def ders4_dizinini_bul():
    """Jupyter kernel'inin çalışma dizini nereden başlatıldığından (repo kökü, Ders4'ün
    kendisi vb.) bağımsız olarak Ders4 klasörünü bulur, böylece göreli yollar her zaman
    doğru çalışır."""
    isaret_dosya = "benchmark_soru_havuzu.json"
    baslangic = os.getcwd()

    adaylar = [baslangic, os.path.join(baslangic, "Ders4")]
    ust = baslangic
    for _ in range(5):
        ust = os.path.dirname(ust)
        adaylar.append(os.path.join(ust, "Ders4"))

    for aday in adaylar:
        if os.path.exists(os.path.join(aday, isaret_dosya)):
            return aday

    raise FileNotFoundError(
        f"'{isaret_dosya}' bulunamadı. Notebook'u 'Ders4' klasöründen ya da onu içeren "
        f"proje kökünden çalıştırdığından emin ol (şu an çalışma dizini: {baslangic})."
    )


os.chdir(ders4_dizinini_bul())
print("Çalışma dizini:", os.getcwd())

VERI_KLASORU = "../Ders2/DataCollection-Scrapping/json_ciktilari"
ENV_DOSYASI = "../Ders2/DataCollection-Scrapping/.env"

HAVUZ_DOSYASI = "benchmark_soru_havuzu.json"
LLM_SONUC_DOSYASI = "llm_coktan_secmeli_ham_v2.json"
COKTAN_SECMELI_DOSYASI = "benchmark_coktan_secmeli.json"

RASTGELE_TOHUM = 42


def env_dosyasini_yukle(yol):
    if not os.path.exists(yol):
        return
    with open(yol, "r", encoding="utf-8") as f:
        for satir in f:
            satir = satir.strip()
            if not satir or satir.startswith("#") or "=" not in satir:
                continue
            anahtar, _, deger = satir.partition("=")
            os.environ.setdefault(anahtar.strip(), deger.strip().strip('"').strip("'"))


env_dosyasini_yukle(ENV_DOSYASI)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-3.5-sonnet")

print("Model:", OPENROUTER_MODEL, "| API key tanımlı mı:", bool(OPENROUTER_API_KEY))


## 1. Soru Havuzunu Yükleme, Ürün Bilgisiyle Birleştirme ve Son Temizlik

In [ ]:
def urun_aciklamalarini_yukle(klasor):
    aciklamalar = {}
    for dosya_adi in sorted(os.listdir(klasor)):
        if not dosya_adi.endswith(".json"):
            continue
        urun_id = dosya_adi.replace(".json", "")
        with open(os.path.join(klasor, dosya_adi), "r", encoding="utf-8") as f:
            diyaloglar = json.load(f)
        sistem_mesaji = diyaloglar[0][0]["content"]
        aciklamalar[urun_id] = sistem_mesaji.split("Ürün Özellikleri:", 1)[-1].strip()
    return aciklamalar


with open(HAVUZ_DOSYASI, "r", encoding="utf-8") as f:
    havuz = json.load(f)

urun_aciklamalari = urun_aciklamalarini_yukle(VERI_KLASORU)
for kayit in havuz:
    kayit["urun_aciklamasi"] = urun_aciklamalari.get(kayit["urun_id"], "")

havuz_df = pd.DataFrame(havuz)
print(f"Havuzdaki toplam soru: {len(havuz_df)}")


In [ ]:
# Analiz notebook'unda LLM'in eleyemediği kalıntı "gerçek bilgi içermeyen" cevapları
# (mesai saati / açıklamaya yönlendirme / "bilgi bulunamadı" gibi) son bir kez ayıklıyoruz.
# Bu tür cevaplardan doğru bir "kibar ve ikna edici" şık üretilemez, çünkü ortada
# aktarılacak somut bir bilgi yoktur.
KALINTI_SABLON_KELIMELERI = [
    "hafta sonu", "hafta içi", "haftaiçi", "çalışma saatlerimiz",
    "resmi tatil", "bayram", "geçmiş mesajları gör",
    "rastlanmadı", "bulunamadı",
    "açıklama kısmında yer almaktadır", "açıklama kısmında detaylı",
    "açıklamada yer al", "açıklama kısmına",
]


def kalinti_sablon_mu(cevap):
    cevap_kucuk = cevap.lower()
    return any(k in cevap_kucuk for k in KALINTI_SABLON_KELIMELERI)


havuz_df["kalinti_sablon"] = havuz_df["referans_cevap"].apply(kalinti_sablon_mu)
print(f"Elenen kalıntı şablon/bilgisiz cevap: {havuz_df['kalinti_sablon'].sum()} / {len(havuz_df)}")

temiz_havuz_df = havuz_df[~havuz_df["kalinti_sablon"]].reset_index(drop=True)
print(f"Çoktan seçmeliye çevrilecek soru sayısı: {len(temiz_havuz_df)}")


## 2. LLM İle Soru Zeminlemesi + Doğru Şık + 3 Yanlış Şık Üretimi

Her soru için LLM'e ürün bilgisi, soru ve ham (gerçek) cevap veriliyor; model şunu üretiyor:
- `soru_final`: GERÇEK CEVAP'taki bilgi ürün bilgisinden çıkarılamıyorsa (kargo firması,
  koli/desi, spesifik bir parça ölçüsü gibi satıcıya özgü bilgiler), bu bilgi alıcının
  zaten öğrenmiş/duymuş olduğu doğal bir cümleyle orijinal soruya eklenir. Bilgi zaten ürün
  bilgisinden çıkarılabiliyorsa `soru_final` orijinal soruyla aynı kalır. Bu sayede test
  edilen model, elinde olmayan bir bilgiyi tahmin etmek zorunda kalmadan, sadece verilen
  bağlamı (ürün bilgisi + soru) doğru işleyip işlemediği üzerinden değerlendirilir.
- `dogru_cevap`: ham cevaptaki bilgiyle birebir tutarlı, kibar ve (bilgi olumsuzsa bile)
  müşteriyi ikna etmeye/elde tutmaya çalışan bir üslupla yeniden yazılmış hali.
- `yanlis_bilgi`: aynı kibar üslupta ama gerçek bilgiyle çelişen yanlış bir cevap.
- `kaba_dogru_bilgi`: doğru bilgiyi içeren ama kaba/soğuk, ikna çabası olmayan bir cevap.
- `alakasiz_sablon`: soruyu hiç cevaplamayan, veri setindeki gerçek kalıp mesajlara benzer
  otomatik/şablon bir cevap.


In [ ]:
SABLON_ORNEKLERI = """- "Merhaba, hafta sonu çalışmamız yoktur. Sizlere yardımcı olabilmemiz adına hafta içi \
mesajınızı tekrar iletmeniz halinde tarafınıza yardımcı olunacaktır. Anlayışınız için teşekkür \
eder, sağlıklı günler dileriz."
- "Merhaba, haftaiçi çalışma saatlerimiz 08:00-18:00 arasıdır. Almak istediğiniz bilgiler ya da \
talepleriniz için, haftaiçi belirtilen saatler aralığında mesajınızı tekrar iletmenizi rica ederiz."
- "Merhaba, ürün ölçü/malzeme bilgileri açıklama kısmında yer almaktadır. Detaylı olarak açıklama \
kısmında inceleyebilirsiniz." """

LLM_SISTEM_PROMPTU = f"""Sen bir e-ticaret satıcı asistanı benchmark'ı hazırlayan bir analistsin. \
Sana bir ürünün özellikleri, bir alıcı sorusu ve bu soruya verilmiş GERÇEK ve DOĞRU bir cevap \
(ham/kısa haliyle) verilecek. Görevin bu bilgiden yola çıkarak çoktan seçmeli bir benchmark \
sorusu için soru zeminlemesi ve 1 doğru, 3 yanlış şık üretmek.

ÖNCE soru_final alanını üret:
- GERÇEK CEVAP'taki bilgi (ör. kargo firması adı, sevkiyat ili, koli/desi bilgisi, spesifik \
  bir parça ölçüsü, montaj detayı gibi satıcıya özgü bilgiler) yukarıdaki ÜRÜN ÖZELLİKLERİ \
  metninden doğrudan ya da dolaylı çıkarılamıyorsa: orijinal soruyu, alıcı bu bilgiyi zaten \
  bir yerden (ürün sayfası, kargo takip ekranı, önceki bir siparişi, bir yorum vb.) öğrenmiş/\
  duymuş gibi DOĞAL bir cümleyle genişleterek yeniden yaz; gereken veri (isim, sayı, firma \
  adı vb.) böylece sorunun içinde hazır bulunsun. Cevabı doğrudan tekrar etme, sadece cevaba \
  ulaşmak için gereken veriyi bir gerekçe/bağlam cümlesi içinde ver.
- GERÇEK CEVAP'taki bilgi zaten ÜRÜN ÖZELLİKLERİ metninden çıkarılabiliyorsa, soru_final \
  orijinal soruyla aynı kalır (değiştirme).

SONRA doğru ve yanlış şıkları üret. Şıklar HER ZAMAN soru_final'e cevap gibi görünmeli.

DOĞRU ŞIK (dogru_cevap):
- Verilen gerçek cevaptaki bilgiyle TAMAMEN tutarlı olmalı; yeni/uydurma bir bilgi (ör. var \
  olmayan bir ürün linki, ölçü, tarih) EKLEME.
- Kibar, sıcak ve satış diline uygun bir üslupla yazılmalı.
- Bilgi müşteri için olumsuzsa (ürün/özellik mevcut değilse), bunu nazikçe ifade edip \
  müşteriyi kırmadan, teşekkür ederek veya başka bir konuda yardımcı olabileceğini belirterek \
  ilgisini canlı tutmaya çalış; ama VERİDE OLMAYAN somut bir alternatif ürün/link uydurma.

YANLIŞ ŞIKLAR (üçü de soru_final'e cevap gibi görünmeli, birbirinden farklı biçimde yanlış olmalı):
1. yanlis_bilgi: Doğru şıkla aynı kibar üslupta ama gerçek cevapla ÇELİŞEN yanlış bir bilgi \
   içermeli (örn. "var" yerine "yok" demek, farklı bir ölçü/malzeme/renk söylemek gibi).
2. kaba_dogru_bilgi: Doğru bilgiyi içeriyor ama kaba, soğuk, özensiz, müşteriyi önemsemeyen \
   bir üslupla yazılmalı (kibarlık/ikna çabası YOK, çok kısa ve sert olabilir).
3. alakasiz_sablon: Soruyu hiç cevaplamayan, konuyla ilgisiz otomatik/kalıp bir mesaj olmalı. \
   Veri setindeki gerçek kalıp mesaj örnekleri (stil referansı, birebir kopyalama):
{SABLON_ORNEKLERI}

SADECE aşağıdaki formatta geçerli bir JSON nesnesi döndür, başka açıklama ekleme:
{{"soru_final": "...", "dogru_cevap": "...", "yanlis_bilgi": "...", "kaba_dogru_bilgi": "...", "alakasiz_sablon": "..."}}"""


def llm_kullanici_promptu(satir):
    return (
        f"Ürün Özellikleri:\n{satir['urun_aciklamasi']}\n\n"
        f"Soru: {satir['soru']}\n\n"
        f"Gerçek Cevap (ham veri): {satir['referans_cevap']}"
    )


def json_govdesini_ayikla(metin):
    metin = metin.strip()
    if metin.startswith("```"):
        metin = re.sub(r"^```[a-zA-Z]*\n?", "", metin)
        metin = re.sub(r"```$", "", metin).strip()
    eslesme = re.search(r"\{.*\}", metin, re.DOTALL)
    return eslesme.group(0) if eslesme else metin


In [ ]:
import httpx


async def openrouter_secenek_uret(client, satir, max_deneme=3):
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": LLM_SISTEM_PROMPTU},
            {"role": "user", "content": llm_kullanici_promptu(satir)},
        ],
        "temperature": 0.4,
        "max_tokens": 800,
    }
    url = OPENROUTER_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    son_hata = None
    for deneme in range(1, max_deneme + 1):
        try:
            resp = await client.post(url, json=payload, headers=headers, timeout=90)
            resp.raise_for_status()
            icerik = resp.json()["choices"][0]["message"]["content"]
            sonuc = json.loads(json_govdesini_ayikla(icerik))
            gerekli_alanlar = {"soru_final", "dogru_cevap", "yanlis_bilgi", "kaba_dogru_bilgi", "alakasiz_sablon"}
            if not gerekli_alanlar.issubset(sonuc.keys()):
                raise ValueError(f"Eksik alan(lar): {gerekli_alanlar - sonuc.keys()}")
            return sonuc
        except Exception as e:
            son_hata = str(e)
            await asyncio.sleep(2 * deneme)
    print(f"  Uyarı: şık üretimi başarısız oldu, atlanıyor. Hata: {son_hata}")
    return None


## 3. Tüm Sorular Üzerinde Çalıştırma (Resumable)

In [ ]:
def onceki_sonuclari_yukle():
    if os.path.exists(LLM_SONUC_DOSYASI):
        with open(LLM_SONUC_DOSYASI, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def sonuclari_kaydet(sonuclar):
    tmp = LLM_SONUC_DOSYASI + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(sonuclar, f, ensure_ascii=False, indent=2)
    os.replace(tmp, LLM_SONUC_DOSYASI)


async def tum_sorulari_isle(df, max_eszamanli=10):
    sonuclar = onceki_sonuclari_yukle()
    sem = asyncio.Semaphore(max_eszamanli)
    kilit = asyncio.Lock()
    toplam = len(df)
    tamamlanan = sum(1 for i in range(toplam) if str(i) in sonuclar)

    async with httpx.AsyncClient() as client:
        async def bir_soruyu_isle(i, satir):
            nonlocal tamamlanan
            if str(i) in sonuclar and sonuclar[str(i)]:
                return
            async with sem:
                sonuc = await openrouter_secenek_uret(client, satir)
            async with kilit:
                sonuclar[str(i)] = sonuc
                tamamlanan += 1
                if tamamlanan % 20 == 0 or tamamlanan == toplam:
                    sonuclari_kaydet(sonuclar)
                    print(f"  {tamamlanan}/{toplam} tamamlandı.")

        await asyncio.gather(*(
            bir_soruyu_isle(i, satir) for i, satir in df.iterrows()
        ))

    sonuclari_kaydet(sonuclar)
    return sonuclar


assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY bulunamadı, Ders2/.env dosyasını kontrol et."
llm_sonuclari = await tum_sorulari_isle(temiz_havuz_df)
print("Şık üretimi tamamlandı.")


## 4. Şıkları Karıştırma ve Son Benchmark Formatının Oluşturulması

Doğru cevap ve 3 yanlış şık, her soru için rastgele (sabit `RASTGELE_TOHUM` ile
tekrarlanabilir şekilde) A-D harflerine dağıtılır ve doğru harf ayrıca kaydedilir.

In [ ]:
random.seed(RASTGELE_TOHUM)
HARFLER = ["A", "B", "C", "D"]

coktan_secmeli_havuz = []
for i, satir in temiz_havuz_df.iterrows():
    sonuc = llm_sonuclari.get(str(i))
    if not sonuc:
        continue

    secenek_metinleri = [
        ("dogru", sonuc["dogru_cevap"]),
        ("yanlis_bilgi", sonuc["yanlis_bilgi"]),
        ("kaba_dogru_bilgi", sonuc["kaba_dogru_bilgi"]),
        ("alakasiz_sablon", sonuc["alakasiz_sablon"]),
    ]
    random.shuffle(secenek_metinleri)

    secenekler = {}
    dogru_harf = None
    for harf, (etiket, metin) in zip(HARFLER, secenek_metinleri):
        secenekler[harf] = metin
        if etiket == "dogru":
            dogru_harf = harf

    coktan_secmeli_havuz.append({
        "urun_id": satir["urun_id"],
        "urun_aciklamasi": satir["urun_aciklamasi"],
        "kategori": satir["kategori"],
        "soru": sonuc.get("soru_final") or satir["soru"],
        "soru_orijinal": satir["soru"],
        "secenekler": secenekler,
        "dogru_secenek": dogru_harf,
    })

with open(COKTAN_SECMELI_DOSYASI, "w", encoding="utf-8") as f:
    json.dump(coktan_secmeli_havuz, f, ensure_ascii=False, indent=2)

print(f"Çoktan seçmeli benchmark kaydedildi: {COKTAN_SECMELI_DOSYASI} ({len(coktan_secmeli_havuz)} soru)")


## 5. Örnek İnceleme

In [ ]:
for ornek in random.sample(coktan_secmeli_havuz, min(3, len(coktan_secmeli_havuz))):
    print("Ürün:", ornek["urun_id"], "| Kategori:", ornek["kategori"])
    print("Soru:", ornek["soru"])
    for harf, metin in ornek["secenekler"].items():
        isaret = " <-- DOĞRU" if harf == ornek["dogru_secenek"] else ""
        print(f"  {harf}) {metin}{isaret}")
    print("-" * 80)

pd.DataFrame(coktan_secmeli_havuz).groupby("urun_id").size().rename("adet")


## 6. Zeminleme Doğrulaması ve Onarımı

`soru_final` üretimi tek bir LLM çağrısında istendiği gibi, doğru şıktaki HER somut bilginin
(ölçü, kargo firması adı vb.) ürün bilgisinde ya da soruda geçtiğinden emin olmayı da
içeriyordu; ama modelin bu kuralı her zaman uygulamadığı gözlemlendi. Bu yüzden otomatik bir
kontrol ekliyoruz: doğru şıktaki sayılar ve bilinen marka/şehir isimleri, ürün bilgisi + soru
metninde geçmiyorsa o soru "zeminlenmemiş" sayılır. Zeminlenmemiş sorular için sadece `soru`
alanını düzelten hedefli, ucuz bir ek LLM turu çalıştırılır (şıklar bozulmaz).

In [ ]:
MARKA_SEHIR_LISTESI = [
    "MNG", "Horoz", "CEVA", "Aras", "Yurtiçi", "PTT", "Sürat", "Kargo", "Lojistik",
    "İstanbul", "Ankara", "İzmir", "Kayseri", "Mardin", "Azerbaycan", "Almanya", "Bursa", "Adana",
]


def sayilar(metin):
    return set(re.findall(r"\d+[.,]?\d*", metin))


def marka_gecenler(metin):
    return {m for m in MARKA_SEHIR_LISTESI if m.lower() in metin.lower()}


def zeminlenmemis_mi(kayit):
    dogru = kayit["secenekler"][kayit["dogru_secenek"]]
    baglam = kayit["urun_aciklamasi"] + " " + kayit["soru"]
    eksik_sayi = sayilar(dogru) - sayilar(baglam)
    eksik_marka = marka_gecenler(dogru) - marka_gecenler(baglam)
    return bool(eksik_sayi or eksik_marka)


zeminlenmemis_indeksler = [i for i, k in enumerate(coktan_secmeli_havuz) if zeminlenmemis_mi(k)]
print(f"Zeminlenmemiş (soruda/üründe geçmeyen somut bilgi içeren) soru sayısı: "
      f"{len(zeminlenmemis_indeksler)} / {len(coktan_secmeli_havuz)}")


In [ ]:
ONARIM_SISTEM_PROMPTU = """Sen bir çoktan seçmeli e-ticaret sorusunu düzenleyen bir editörsün. \
Sana bir ürünün özellikleri, bir soru ve bu sorunun DOĞRU kabul edilen cevabı verilecek. \
Doğru cevaptaki HER somut bilgi (sayı, ölçü, marka/kargo firması adı, şehir vb.) ürün \
özelliklerinde ya da soru metninde ZATEN geçiyor mu kontrol et.

Eğer doğru cevaptaki bir somut bilgi ne ürün özelliklerinde ne de soruda geçmiyorsa, SADECE \
soruyu, alıcı bu bilgiyi zaten bir yerden (ürün sayfası, kargo takip ekranı, bir yorum, önceki \
bir siparişi vb.) öğrenmiş/duymuş gibi doğal bir cümleyle bu somut bilgiyi (sayıyı/adı) İÇİNE \
alacak şekilde yeniden yaz. Cevabı bozma veya tekrar etme, sadece soruyu düzelt.

SADECE aşağıdaki formatta geçerli bir JSON nesnesi döndür, başka açıklama ekleme:
{"soru_duzeltilmis": "..."}"""


def onarim_kullanici_promptu(kayit):
    dogru = kayit["secenekler"][kayit["dogru_secenek"]]
    return (
        f"Ürün Özellikleri:\n{kayit['urun_aciklamasi']}\n\n"
        f"Soru: {kayit['soru']}\n\n"
        f"Doğru Kabul Edilen Cevap: {dogru}"
    )


async def openrouter_soru_onar(client, kayit, max_deneme=3):
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": ONARIM_SISTEM_PROMPTU},
            {"role": "user", "content": onarim_kullanici_promptu(kayit)},
        ],
        "temperature": 0.2,
        "max_tokens": 300,
    }
    url = OPENROUTER_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    son_hata = None
    for deneme in range(1, max_deneme + 1):
        try:
            resp = await client.post(url, json=payload, headers=headers, timeout=60)
            resp.raise_for_status()
            icerik = resp.json()["choices"][0]["message"]["content"]
            sonuc = json.loads(json_govdesini_ayikla(icerik))
            if "soru_duzeltilmis" not in sonuc:
                raise ValueError("soru_duzeltilmis alanı eksik")
            return sonuc["soru_duzeltilmis"]
        except Exception as e:
            son_hata = str(e)
            await asyncio.sleep(2 * deneme)
    print(f"  Uyarı: soru onarımı başarısız oldu, atlanıyor. Hata: {son_hata}")
    return None


async def zeminlenmemis_sorulari_onar(havuz, indeksler, max_eszamanli=10):
    sem = asyncio.Semaphore(max_eszamanli)

    async with httpx.AsyncClient() as client:
        async def bir_kaydi_onar(i):
            async with sem:
                duzeltilmis = await openrouter_soru_onar(client, havuz[i])
            if duzeltilmis:
                havuz[i]["soru"] = duzeltilmis

        await asyncio.gather(*(bir_kaydi_onar(i) for i in indeksler))


if zeminlenmemis_indeksler:
    await zeminlenmemis_sorulari_onar(coktan_secmeli_havuz, zeminlenmemis_indeksler)

    kalan_zeminlenmemis = [i for i, k in enumerate(coktan_secmeli_havuz) if zeminlenmemis_mi(k)]
    print(f"Onarımdan sonra hâlâ zeminlenmemiş: {len(kalan_zeminlenmemis)} / {len(coktan_secmeli_havuz)}")

    with open(COKTAN_SECMELI_DOSYASI, "w", encoding="utf-8") as f:
        json.dump(coktan_secmeli_havuz, f, ensure_ascii=False, indent=2)
    print(f"Güncellenmiş benchmark kaydedildi: {COKTAN_SECMELI_DOSYASI}")
else:
    print("Zeminlenmemiş soru bulunamadı, onarım gerekmedi.")
